# Adaptive displacement-controlled penalty contact

This example demonstrates a two-dimensional frictionless Hertz-type contact
problem using the `ContactGap` weak-form operator.

The two bodies are initially tangent: there is no prescribed initial
penetration. Contact is generated by gradually moving the upper body downward.

The nonlinear solution uses **adaptive displacement stepping**:

1. propose a displacement increment,
2. construct a linear elastic predictor that satisfies the new prescribed
   displacement,
3. iterate the contact equilibrium with a frozen-contact tangent,
4. accept the load step if the nonlinear iteration converges,
5. otherwise return to the last converged state and retry with a smaller
   displacement increment.

This simple cutback strategy prevents a large first step from opening the
contact completely and provides a useful basis for more general nonlinear
solution procedures.


In [102]:
using LowLevelFEM, LinearAlgebra


## Geometry

The geometry contains two initially tangent elastic bodies. The lower straight
boundary is the slave contact surface and the lower circular boundary of the
upper body is the master surface.


In [103]:
openGeometry("2D-Hertz.geo")
# openPreProcessor()


## Material and displacement field


In [104]:
matU = Material("upper", E=2e5, ν=0.3)
matL = Material("lower", E=2e5, ν=0.3)

U = Field(
    [matU, matL],
    type=:VectorField,
    dim=2,
    fieldName=:u
)


Problem("2D-Hertz", :VectorField, 2, 2, Material[Material("upper", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0), Material("lower", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 917, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :u, :rhs, false)

## Linear elastic operator

Both bodies are modeled in plane strain. No external force is applied; the
contact is driven entirely by the prescribed downward displacement of the
upper body.


In [105]:
K =
    ∫(ε(U) ⋅ D(:PlaneStrain, matL) ⋅ ε(U), Ω="lower") +
    ∫(ε(U) ⋅ D(:PlaneStrain, matU) ⋅ ε(U), Ω="upper")

# A zero external load field with the correct finite-element metadata.
f = ∫(U ⋅ [0, 0], Γ="top")


nodal VectorField
[0.0; 0.0; … ; 0.0; 0.0;;]

## Initial state and supports

At zero indentation the top boundary is fixed vertically at its original
position. The horizontal constraints remove rigid-body motion, while the lower
body is supported at its bottom boundary.


In [106]:
support0 = [
    BoundaryCondition("top",    ux=0, uy=0),
    BoundaryCondition("bottom", uy=0),
    BoundaryCondition("left",   ux=0),
    BoundaryCondition("right",  ux=0)
]

u0 = solveField(K, f, support=support0)

showDoFResults(
    u0,
    name="initial displacement",
    factor=1,
    visible=false
)


0

## Contact pair

The master normal must point from the upper body toward the lower body. For
this geometry this requires `normal_sign=-1`.

The signed normal gap convention is

$$
g_n < 0 \quad \text{penetration},
\qquad
g_n > 0 \quad \text{open gap}.
$$

Penalty contact is unilateral: it resists penetration but does not pull
separated surfaces together.


In [107]:
C = contact(
    U,
    master="master",
    slave="slave",
    displacement=u0,
    normal_sign=-1
)

G = ContactGap(C)

# Absolute nodal reference positions. ContactGap acts on r + u.
r = nodePositionVector(U)
r = projectTo2D(r)

# Penalty stiffness.
cn = 1e5


100000.0

The initial gap should be zero at the first contact point and positive away
from it.


In [108]:
updateContact!(C, u0)

gap0 = ContactGap(C, u0; gauss=4)

plotOnBeam(
    "slave",
    nodesToElements(gap0),
    name="initial gap"
)


1

## Nonlinear contact iteration

For a frozen current contact geometry,

$$
R(u)
=
K u + K_c(u)(r+u),
$$

and the correction is obtained from

$$
(K+K_c)\Delta u=-R.
$$

The correction has homogeneous essential boundary conditions. Therefore the
prescribed indentation introduced by the load predictor is preserved during
the contact iterations.

A relaxed update is used,

$$
u_{i+1}=u_i+\omega\Delta u_i,
$$

which is robust for the frozen-geometry contact tangent used here.


In [109]:
# Homogeneous boundary conditions for nonlinear corrections.
support_increment = [
    BoundaryCondition("top",    ux=0, uy=0),
    BoundaryCondition("bottom", uy=0),
    BoundaryCondition("left",   ux=0),
    BoundaryCondition("right",  ux=0)
]

free = freeDoFs(U, support_increment)

# Norm restricted to unconstrained displacement DOFs.
freeNorm(v::VectorField) =
    LinearAlgebra.norm(elementsToNodes(v).a[free, 1])

# Boundary conditions for one displacement-controlled load increment.
function loadIncrementSupport(Δindent)
    return [
        BoundaryCondition("top",    ux=0, uy=-Δindent),
        BoundaryCondition("bottom", uy=0),
        BoundaryCondition("left",   ux=0),
        BoundaryCondition("right",  ux=0)
    ]
end


loadIncrementSupport (generic function with 1 method)

## Adaptive load-step parameters

`indentation_final` is the total downward displacement prescribed at the top
of the upper body.

A successful step can increase the next increment. A failed step is discarded
and retried with a smaller increment.


In [110]:
indentation_final = 0.40

Δindent = 0.02
Δindent_min = 0.00125
Δindent_max = 0.04

cutback = 0.5
growth = 1.5

maxiter = 30
tol_R = 1e-3
tol_u = 1e-3

# Fixed relaxation for the frozen-contact iteration.
ω = 0.5

# Contact quadrature order increment.
contact_gauss = 2

# Iteration counts used to adapt the next load increment.
fast_step = 6
slow_step = 15


15

## Adaptive displacement-controlled solution

Each trial load step starts from the **last converged state**.

A linear elastic predictor introduces the requested displacement increment.
Because this predictor satisfies the new Dirichlet conditions exactly, all
subsequent nonlinear corrections can use homogeneous boundary conditions.

The step is rejected if:

- the contact becomes completely inactive during the nonlinear correction,
- a non-finite residual or correction is encountered, or
- the maximum nonlinear iteration count is reached.

After a rejection, the previous converged solution is restored automatically
and the displacement increment is reduced.


In [111]:
u_committed = copy(u0)
indentation = 0.0

history = NamedTuple[]
attempt = 0
max_attempts = 100

while indentation < indentation_final - 10eps(Float64)

    attempt += 1
    attempt <= max_attempts ||
        error("Adaptive contact solution exceeded the maximum number of attempts.")

    # Do not step beyond the requested final indentation.
    Δtrial = min(Δindent, indentation_final - indentation)
    indentation_trial = indentation + Δtrial

    println()
    println(
        "Attempt ", attempt,
        ": indentation = ", indentation_trial,
        ", increment = ", Δtrial
    )

    # ------------------------------------------------------------------
    # 1. Elastic predictor
    #
    # The predictor changes only the prescribed displacement level.
    # Starting from u_committed keeps all previously converged deformation.
    # ------------------------------------------------------------------
    Δu_load = solveField(
        K,
        f,
        support=loadIncrementSupport(Δtrial)
    )

    u_it = u_committed + Δu_load

    Rref = 0.0
    step_converged = false
    step_failed = false
    failure_reason = ""
    iter_used = maxiter
    final_err_R = Inf
    final_err_u = Inf

    for iter in 1:maxiter

        # --------------------------------------------------------------
        # 2. Contact tangent at the current trial configuration
        #
        # updateFrom performs only the geometry update needed by the
        # Gauss-point contact integration.
        # --------------------------------------------------------------
        Kc = ∫(
            G ⋅ cn ⋅ G;
            updateFrom=u_it,
            gauss=contact_gauss,
            threads=:auto
        )

        kc_norm = LinearAlgebra.norm(Kc.A.nzval)

        # If contact existed after the predictor but disappears completely
        # during the nonlinear correction, reject the load step.
        if kc_norm == 0.0
            if iter == 1
                failure_reason =
                    "no active Gauss-point contact after the load predictor"
            else
                failure_reason =
                    "contact opened completely during the nonlinear correction"
            end

            step_failed = true
            break
        end

        # --------------------------------------------------------------
        # 3. Residual
        # --------------------------------------------------------------
        rc = Kc * (r + u_it)
        R = K * u_it - f + rc

        Rnorm = freeNorm(R)

        if !isfinite(Rnorm)
            failure_reason = "non-finite residual"
            step_failed = true
            break
        end

        if iter == 1
            Rref = max(Rnorm, eps(Float64))
        end

        # --------------------------------------------------------------
        # 4. Frozen-contact correction
        #
        # The correction has zero values on all prescribed DOFs, so the
        # displacement level imposed by the predictor is retained.
        # --------------------------------------------------------------
        Δu = solveField(
            K + Kc,
            -R,
            support=support_increment
        )

        Δunorm = freeNorm(Δu)

        if !isfinite(Δunorm)
            failure_reason = "non-finite displacement correction"
            step_failed = true
            break
        end

        err_R = Rnorm / Rref
        err_u =
            Δunorm /
            max(freeNorm(u_it), eps(Float64))

        final_err_R = err_R
        final_err_u = err_u

        println(
            "  iter = ", iter,
            ", |R|/|R0| = ", err_R,
            ", |du|/|u| = ", err_u
        )

        # --------------------------------------------------------------
        # 5. Convergence test
        # --------------------------------------------------------------
        if err_R < tol_R && err_u < tol_u
            step_converged = true
            iter_used = iter
            break
        end

        # --------------------------------------------------------------
        # 6. Relaxed nonlinear update
        # --------------------------------------------------------------
        u_it = u_it + ω * Δu
    end

    # ------------------------------------------------------------------
    # 7. Accept or reject the trial load step
    # ------------------------------------------------------------------
    if step_converged

        u_committed = copy(u_it)
        indentation = indentation_trial

        push!(
            history,
            (
                indentation=indentation,
                increment=Δtrial,
                iterations=iter_used,
                residual=final_err_R,
                correction=final_err_u
            )
        )

        println(
            "  ACCEPTED: indentation = ", indentation,
            ", iterations = ", iter_used
        )

        # Increase the next step after easy convergence.
        if iter_used <= fast_step
            Δindent = min(growth * Δtrial, Δindent_max)

        # Reduce the next step after difficult convergence.
        elseif iter_used >= slow_step
            Δindent = max(0.75 * Δtrial, Δindent_min)

        else
            Δindent = Δtrial
        end

    else

        # The committed solution has not been modified, so rollback is
        # automatic. Only the next trial increment has to be reduced.
        Δindent = cutback * Δtrial

        println(
            "  REJECTED: ",
            isempty(failure_reason) ?
                "nonlinear iteration did not converge" :
                failure_reason
        )

        println("  Cutback: new increment = ", Δindent)

        Δindent >= Δindent_min ||
            error(
                "Adaptive contact solution failed: the required load " *
                "increment became smaller than Δindent_min."
            )
    end
end

u = u_committed

println()
println("Adaptive contact solution completed.")
println("Final indentation = ", indentation)
println("Accepted load steps = ", length(history))



Attempt 1: indentation = 0.02, increment = 0.02
  iter = 1, |R|/|R0| = 1.0, |du|/|u| = 0.1554171903311473
  iter = 2, |R|/|R0| = 0.49981204169298127, |du|/|u| = 0.08126517023995115
  iter = 3, |R|/|R0| = 0.2601759141366031, |du|/|u| = 0.05774804624526594
  iter = 4, |R|/|R0| = 0.13304358237062475, |du|/|u| = 0.03506112937133184
  iter = 5, |R|/|R0| = 0.06651983301225838, |du|/|u| = 0.019127378001675098
  iter = 6, |R|/|R0| = 0.03497915883204024, |du|/|u| = 0.011103131206831205
  iter = 7, |R|/|R0| = 0.017489169279506306, |du|/|u| = 0.0055660906029322815
  iter = 8, |R|/|R0| = 0.008744396615424777, |du|/|u| = 0.0027866248418534404
  iter = 9, |R|/|R0| = 0.004372109547192901, |du|/|u| = 0.001394185977907319
  iter = 10, |R|/|R0| = 0.002186012210455442, |du|/|u| = 0.0006973015117879231
  iter = 11, |R|/|R0| = 0.001092985531656446, |du|/|u| = 0.00034869816034047266
  iter = 12, |R|/|R0| = 0.0005464827831970537, |du|/|u| = 0.0001743586467027294
  ACCEPTED: indentation = 0.02, iterations = 

## Load-step history

The table below shows the accepted displacement increments and the number of
nonlinear iterations required by each step. It is useful for tuning the
initial increment, relaxation factor and cutback parameters.


In [112]:
history


20-element Vector{NamedTuple}:
 (indentation = 0.02, increment = 0.02, iterations = 12, residual = 0.0005464827831970537, correction = 0.0001743586467027294)
 (indentation = 0.04, increment = 0.02, iterations = 12, residual = 0.0005336922412204495, correction = 0.00010365167150511741)
 (indentation = 0.06, increment = 0.02, iterations = 12, residual = 0.0005050740248873232, correction = 6.296898278463389e-5)
 (indentation = 0.08, increment = 0.02, iterations = 11, residual = 0.000989818268860531, correction = 9.774932089143827e-5)
 (indentation = 0.1, increment = 0.02, iterations = 12, residual = 0.0005189995480284503, correction = 4.344561823338865e-5)
 (indentation = 0.12000000000000001, increment = 0.02, iterations = 12, residual = 0.0005057974222989758, correction = 3.303957000770969e-5)
 (indentation = 0.14, increment = 0.02, iterations = 11, residual = 0.0009806887324105783, correction = 5.124459755491232e-5)
 (indentation = 0.16, increment = 0.02, iterations = 12, residual = 0.0

## Final contact state

The full nodal contact state is updated only once after the incremental
solution. The Gauss-point gap can then be projected back to the slave
Lagrange space for visualization.

For the penalty law,

$$
p_n = \max(-c_n g_n,0).
$$


In [113]:
# Synchronize nodal contact data for post-processing.
updateContact!(C, u)

gap = ContactGap(C, u)

pressure = mapScalarField(
    g -> max(-cn * g, 0.0),
    gap
)

showDoFResults(
    u,
    name="final displacement",
    factor=1,
    visible=false
)

plotOnBeam(
    "slave",
    nodesToElements(gap),
    name="gap"
)

plotOnBeam(
    "slave",
    nodesToElements(pressure),
    name="contact pressure"
)


4

## Remarks

- The penalty stiffness should be large enough to limit penetration, but an
  excessively large value deteriorates conditioning and can excite
  mesh-scale oscillations.
- Adaptive displacement stepping controls the *loading path*; it does not
  replace a sensible penalty stiffness.
- The present iteration uses a frozen contact geometry and fixed relaxation.
  More advanced variants may introduce a line search, augmented Lagrangian
  enforcement, or a consistent contact tangent.
- For performance studies, the CSC sparsity pattern of `Kc` can also be reused
  between iterations when the current slave-master connectivity remains
  covered by the stored pattern.


In [114]:
openPostProcessor()
